# Indictrans Inference Earlier Version

Earlier IndicTrans2 inference and tokenizer workflow, retained as an earlier implementation.

Stage: earlier experiment.

Uses the [AI4Bharat IndicTrans2 inference workflow](https://github.com/AI4Bharat/IndicTrans2), with file/batch processing experiments. Original explanatory cells are retained below.


# IndicTrans2 HF Inference

We provide an example notebook on how to use our IndicTrans2 models which were originally trained with the fairseq to HuggingFace transformers for inference purpose.


## Setup

Please run the cells below to install the necessary dependencies.


In [ ]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [ ]:
%%capture
%cd /content/IndicTrans2/huggingface_interface

In [ ]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers>=4.33.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

In [ ]:
!ls -a

**IMPORTANT : Restart your run-time first and then run the cells below.**

## Inference


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer
from IndicTransToolkit import IndicProcessor

BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None

In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, direction, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = IndicTransToolkit(direction=direction)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            src=True,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(generated_tokens.detach().cpu().tolist(), src=False)

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

### English to Indic Example


In [ ]:
def convert_to_list_of_strings(input_text):
    # Split the input text by newline characters
    lines = input_text.split('\n')
    lines = lines[1:-1]
    # Return the list of lines
    return lines

# Example input
input_text = """
are the three key actions in the sustainability movement aimed at minimizing waste and conserving resources.
This involves reducing the amount of waste generated in the first place by using fewer resources and consuming less. It encourages people to be mindful of their consumption habits and to make choices that minimize their environmental impact.
Recycling involves collecting materials that would otherwise be considered waste and processing them to create new products.
Reuse encourages finding new purposes for items instead of disposing of them after a single use. This can involve repairing, repurposing, or donating items to extend their lifespan and reduce the need for new products.
Recycle
Reuse
Reduce
Reduce, Reuse, Recycle
"""

# Convert the input text into a list of strings
list_of_strings = convert_to_list_of_strings(input_text)

# Print the list of strings
for line in list_of_strings:
    print(line)


In [ ]:
en_sents = convert_to_list_of_strings("""


Give Them Compliments
Creating Everlasting Memories: Your Ultimate Wedding Planner
Table of content
Attire and beauty
Embark on your wedding planning journey with confidence! 'Creating Everlasting Memories' is your comprehensive guide, providing expert advice and inspiration to craft the wedding day of your dreams
Worksheet to outline the couple's vision, themes, and must-haves
Cost-Saving Strategies
Ceremony Planning and Reception Essentials
We plan your ceremony scripts, vows, and unity options. For the reception, we handle seating, menus, and dessert tables, making your day stress-free
contacts
Thank You
is more tastier when served with Love
When you've decided on your cover story, come up with a list of topics for your feature articles. This can range from interviews, product reviews, human interest pieces, and even lists. Think about what your audience would be interested in and get writing! Again, choose engaging photos and graphics to accompany your words, as these also help catch your audience's eye.
An editor's letter is a message written for a variety of purposes, from friendly to formal. They can help the reader understand the content and creative direction of a certain issue, they can give the reader and idea of what they can find in the magazine, or they can explain the issue's theme and how it can resonate with them. If you’re thinking of writing an editor's letter yourself, make your intentions clear from the start. You can be fun and creative or straightforward, depending on your needs. For the main content of your letter, there are often three main parts: the introduction, main paragraph, and conclusion. Your letter’s introduction can be a brief greeting, a few polite statements, or a background of why you’re writing. The main paragraph is the bulk of your letter, containing the most important parts of your message. Finally, the conclusion sums up all your ideas. It can also include a closing statement or salutation.
Catherine Earnshaw
Recipe Book
Introduction
Grocery Shopping List
Pecan Pie
¼ Melted Butter
Bake 1 hour or until filling is set
Serving
From the kitchen of:
Successful partnerships should be mutually beneficial.
To do so, we uphold our commitment to provide the best automotive content compared to other publications that you come across. Happy reading, everyone!
Date
Signature
Manager
The smart home market is projected to grow by 15% annually, reaching $200 billion by 2025.
Effective marketing strategies are crucial for business growth. Utilizing digital channels, such as social media and email campaigns, enhances brand visibility. Understanding customer needs through data analysis drives personalized experiences. Consistent branding and engaging content foster customer loyalty. Monitoring trends and adapting strategies ensures competitiveness. Invest in creative, targeted marketing to maximize ROI.
Oct-Dec
Create a Content Calendar
Content Strategy
Why Audience Understanding Matters?
Understanding the Target Audience
Threat
Travel Guide
food
""")

In [ ]:
#magazines

en_sents = convert_to_list_of_strings("""
How To Use Perfume Properly
fragrance
Magazine
Perfume New Arrival
The quality of our publication is something that we can’t compromise in any way possible. The as accurate and substantial publication is the key for the people to receive the most accurate automotive-related information, it is our responsibility to do just that.
  Note
To do so, we uphold our commitment to provide the best automotive content compared to other publications that you come across. Happy reading, everyone!
Grab It Fast and Don’t Miss Out
Having a consistently upgraded newsletter about your brand can keep your customers’ long-term interest and maintain the strong relationship that you have developed with your customers over the years.
An editor's letter is a message written for a variety of purposes, from friendly to formal.
Inspiring Creations from Every Medium
Publication
Aaron Loeb
Date
travel Diaries
An editor's letter is a message written for a variety of purposes, from friendly to formal. They can help the reader understand the content and creative direction of a certain issue, they can give the reader and idea of what they can find in the magazine, or they can explain the issue's theme and how it can resonate with them. If you’re thinking of writing an editor's letter yourself, make your intentions clear from the start. You can be fun and creative or straightforward, depending on your needs. For the main content of your letter, there are often three main parts: the introduction, main paragraph, and conclusion. Your letter’s introduction can be a brief greeting, a few polite statements, or a background of why you’re writing. The main paragraph is the bulk of your letter, containing the most important parts of your message. Finally, the conclusion sums up all your ideas. It can also include a closing statement or salutation.
When you've decided on your cover story, come up with a list of topics for your feature articles. This can range from interviews, product reviews, human interest pieces, and even lists. Think about what your audience would be interested in and get writing! Again, choose engaging photos and graphics to accompany your words, as these also help catch your audience's eye.
# After writing all your articles and adding them to your layout, list down all the titles to set up your table of contents. You can add a brief description for each article or keep it simple and paste the feature titles on the page. Don't forget your page numbers too!
Hawaii Tropical Bliss
Captured in a moment of creativity and collaboration. A big thank you to the entire production team for their dedication and hard work!
"Success is not final, failure is not fatal: It is the courage to continue that counts." — Winston Churchill
From hidden gems to iconic landmarks, each issue is a curated guide to inspire your next adventure.
Table of Contents
Index
Discover Seaside Hotel in Ibiza
Managing Editor
Advertisement
Book Now
""")


In [ ]:
# #FORMS

# en_sents = convert_to_list_of_strings("""
# Full name
# Address
# Date of Birth
# Email
# Gender
# Marital status
# Feedback form
# summary
# notes
# value for money
# recommendation
# Country
# National ID no.
# signature
# male
# female
# current place
# Nationality
# Domicile
# Father's Name
# Emergency contact
# How do you learn best?
# This space is where you can share information on the section, such us: topic, discussion points, goals and activities.
# Did you find what you were looking for today? (Yes/No)
# How would you rate our customer service on a scale of 1-5?
# In which department do you work?
# What is your company's annual revenue?
# How likely are you to recommend us to a friend or colleague?
# Post code
# Are you coming to the party?
# Why not watch a movie if you’re bored?
# Do you know whether James has finished the report or not?
# Can I ask whether you have received your handouts?
# How many people died of disease or other non-combat causes during the Civil War?
# When did Abraham Lincoln die?
# May I ask why you chose that vendor?
# If you could change one thing about the training your end users receive, what would it be?
# If the problem you have described is not resolved, how will it impact your organization’s sales?
# If your end users were thoroughly trained and using this platform effectively, how could that increase company productivity?
# Phone number
# Sunday
# January
# Date
# The things you can appreciate:
# Registration form
# treatment
# value
# Neval street, colony 5, street 12346
# Information
# photo
# homework tracking
# student information
# mother's name
# emergency telephone
# """)

In [ ]:
# en_sents = convert_to_list_of_strings("""
# For the world's hungry and unsustainable solutions, renewable energy
# Minimalist and modern interiors
# vanilla dream cupcakes
# renewable energy
# 4:00 p.m. - Lecture by Sandra Haro
# A guide to sustainable solutions for a greener future.
# The Energy Landscape of Tomorrow
# At Lara's Bakery, we orchestrate a suite symphony that takes Johnny's taste buds to new heights. Immerse yourself in a world of ecstasy, where every creation is made of high-quality ingredients.
# Baby spa and massage.
# Explore and discover
# pause
# call now
# Where are we going?
# How should we take care?
# Protective force
# Full of the essence of fresh oranges.
# visit us
# values
# love
# Full of the essence of fresh oranges.
# """)

In [ ]:
# en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"  # ai4bharat/indictrans2-en-indic-dist-200M
# en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

# ip = IndicProcessor(inference=True)


# print(en_sents)
# # # Strip newline characters from each line
# en_sents = [line.strip() for line in en_sents]
# print(en_sents)
# # Define source and target languages
# src_lang, tgt_lang = "eng_Latn", "kas_Arab"

# # Translate sentences
# other_translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

# # Save the translations to <language>.txt
# with open(f'{tgt_lang}.txt', 'w') as out_file:
#     for translation in other_translations:
#         out_file.write(translation + '\n')

# # Flush the models to free GPU memory
# del en_indic_tokenizer, en_indic_model


In [ ]:
#for Synthetic doc generation
en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"  # ai4bharat/indictrans2-en-indic-dist-200M
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

import pandas as pd
from tabulate import tabulate

# en_sents = [
#     "When I was young, I used to go to the park every day.",
#     "He has many old books, which he inherited from his ancestors.",
#     "I can't figure out how to solve my problem.",
#     "She is very hardworking and intelligent, which is why she got all the good marks.",
#     "We watched a new movie last week, which was very inspiring.",
#     "If you had met me at that time, we would have gone out to eat.",
#     "She went to the market with her sister to buy a new sari.",
#     "Raj told me that he is going to his grandmother's house next month.",
#     "All the kids were having fun at the party and were eating lots of sweets.",
#     "My friend has invited me to his birthday party, and I will give him a gift.",
# ]

src_lang, tgt_lang = "eng_Latn" , "sat_Olck"
hi_translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(en_sents, hi_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{translation}")

# Add underscore line under each pair of sentences
data_with_lines = []
for en, hi in zip(en_sents, hi_translations):
    data_with_lines.append((en, hi))
    data_with_lines.append(("_" * len(en), "_" * len(hi)))  # Add lines for separation

# flush the models to free the GPU memory
del en_indic_tokenizer, en_indic_model

### Indic to English Example

In [ ]:

en_indic_ckpt_dir = "ai4bharat/indictrans2-en-indic-1B"  # ai4bharat/indictrans2-en-indic-dist-200M
en_indic_tokenizer, en_indic_model = initialize_model_and_tokenizer(en_indic_ckpt_dir, "en-indic", quantization)

ip = IndicProcessor(inference=True)

import pandas as pd
from tabulate import tabulate



src_lang, tgt_lang = "brx_Deva" , "eng_Latn"
hi_translations = batch_translate(en_sents, src_lang, tgt_lang, en_indic_model, en_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(en_sents, hi_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{translation}")

# Add underscore line under each pair of sentences
data_with_lines = []
for en, hi in zip(en_sents, hi_translations):
    data_with_lines.append((en, hi))
    data_with_lines.append(("_" * len(en), "_" * len(hi)))  # Add lines for separation

# flush the models to free the GPU memory
del en_indic_tokenizer, en_indic_model

In [ ]:
hi_sents = convert_to_list_of_strings("""
"ꯅꯦꯚꯜ ꯁ ꯭ ꯇ ꯭ ꯔꯤꯠ, ꯀꯣꯂꯣꯅꯤ 5, ꯁ ꯭ ꯇꯦꯟ 12346"
ꯀꯔꯤꯒꯨꯝꯕ ꯅꯍꯥꯛꯅ ꯅꯍꯥꯛꯀꯤ ꯑꯔꯣꯏꯕ ꯁꯤꯖꯤꯟꯅꯔꯤꯕꯁꯤꯡꯅ ꯐꯪꯂꯤꯕ ꯇ ꯭ ꯔꯦꯅꯤꯡ ꯑꯗꯨꯒꯤ ꯃꯇꯥꯡꯗ ꯋꯥꯐꯝ ꯑꯃ ꯍꯣꯡꯗꯣꯛꯄ ꯉꯝꯂꯕꯗꯤ, ꯃꯗꯨ ꯀꯔꯤ ꯑꯣꯏꯒꯗꯒꯦ?
""")

In [ ]:
indic_en_ckpt_dir = "ai4bharat/indictrans2-indic-en-1B"  # ai4bharat/indictrans2-indic-en-dist-200M
indic_en_tokenizer, indic_en_model = initialize_model_and_tokenizer(indic_en_ckpt_dir, "indic-en", quantization)

ip = IndicProcessor(inference=True)



src_lang, tgt_lang = "brx_Deva", "eng_Latn"
en_translations = batch_translate(hi_sents, src_lang, tgt_lang, indic_en_model, indic_en_tokenizer, ip)


print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(hi_sents, en_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

# flush the models to free the GPU memory
del indic_en_tokenizer, indic_en_model

### Indic to Indic Example


In [ ]:
indic_indic_ckpt_dir = "ai4bharat/indictrans2-indic-indic-1B"  # ai4bharat/indictrans2-indic-indic-dist-320M
indic_indic_tokenizer, indic_indic_model = initialize_model_and_tokenizer(indic_indic_ckpt_dir, "indic-indic", quantization)

ip = IndicProcessor(inference=True)

hi_sents = [
    "जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।",
    "उसके पास बहुत सारी पुरानी किताबें हैं, जिन्हें उसने अपने दादा-परदादा से विरासत में पाया।",
    "मुझे समझ में नहीं आ रहा कि मैं अपनी समस्या का समाधान कैसे ढूंढूं।",
    "वह बहुत मेहनती और समझदार है, इसलिए उसे सभी अच्छे मार्क्स मिले।",
    "हमने पिछले सप्ताह एक नई फिल्म देखी जो कि बहुत प्रेरणादायक थी।",
    "अगर तुम मुझे उस समय पास मिलते, तो हम बाहर खाना खाने चलते।",
    "वह अपनी दीदी के साथ बाजार गयी थी ताकि वह नई साड़ी खरीद सके।",
    "राज ने मुझसे कहा कि वह अगले महीने अपनी नानी के घर जा रहा है।",
    "सभी बच्चे पार्टी में मज़ा कर रहे थे और खूब सारी मिठाइयाँ खा रहे थे।",
    "मेरे मित्र ने मुझे उसके जन्मदिन की पार्टी में बुलाया है, और मैं उसे एक तोहफा दूंगा।",
]
src_lang, tgt_lang = "hin_Deva", "brx_Deva"
mr_translations = batch_translate(hi_sents, src_lang, tgt_lang, indic_indic_model, indic_indic_tokenizer, ip)

print(f"\n{src_lang} - {tgt_lang}")
for input_sentence, translation in zip(hi_sents, mr_translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")

# flush the models to free the GPU memory
del indic_indic_tokenizer, indic_indic_model